# Comparing Clustering Algorithms

## Learning Objectives

By the end of this notebook, you will be able to:
- Compare DBSCAN with K-Means, OPTICS, and Hierarchical clustering
- Understand the strengths and weaknesses of each algorithm
- Identify which algorithm is best suited for different data characteristics
- Evaluate clustering quality using quantitative metrics
- Recognize failure cases for each algorithm type

## Prerequisites

- Understanding of DBSCAN basics (see `01_dbscan_basics.ipynb`)
- Familiarity with clustering concepts
- Basic knowledge of K-Means algorithm

## Paper References

This notebook covers concepts from:
- Section 2: Related Work [Paper p.227]
- Section 7: Comparison with other algorithms

---

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.cluster import KMeans, OPTICS, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.datasets import make_moons, make_circles, make_blobs
import sys
sys.path.append('..')

from src.dbscan_from_scratch import DBSCAN
from src.visualization import DBSCANVisualizer
from src.data_loader import load_sample_data

# Initialize visualizer
viz = DBSCANVisualizer(figsize=(15, 5))

print("✓ All imports successful")

## Part 1: Algorithm Overview

### Algorithms We'll Compare

1. **DBSCAN** (Density-Based Spatial Clustering of Applications with Noise)
   - Density-based clustering
   - Finds arbitrary-shaped clusters
   - Identifies noise points
   - Parameters: eps, min_pts

2. **K-Means**
   - Centroid-based clustering
   - Assumes spherical clusters
   - Requires number of clusters k
   - Fast but limited to convex shapes

3. **OPTICS** (Ordering Points To Identify the Clustering Structure)
   - Extension of DBSCAN
   - Handles varying densities
   - Creates reachability plot
   - Parameters: min_samples, xi

4. **Hierarchical Clustering** (Agglomerative)
   - Builds hierarchy of clusters
   - Can use different linkage methods
   - Requires number of clusters or distance threshold
   - Good for nested cluster structures

## Part 2: Comparison on Non-Convex Shapes (Moons Dataset)

The moons dataset contains two interleaving half-circles. This tests how well algorithms handle non-convex cluster shapes.

In [ ]:
# Generate moons dataset
X_moons, _ = make_moons(n_samples=300, noise=0.05, random_state=42)

print(f"Dataset shape: {X_moons.shape}")
print(f"Feature range: [{X_moons.min():.2f}, {X_moons.max():.2f}]")

In [ ]:
# Run all algorithms and measure runtime
results_moons = {}
runtimes_moons = {}

# DBSCAN
start = time.time()
dbscan = DBSCAN(eps=0.3, min_pts=5)
results_moons['DBSCAN'] = dbscan.fit_predict(X_moons)
runtimes_moons['DBSCAN'] = time.time() - start

# K-Means
start = time.time()
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
results_moons['K-Means'] = kmeans.fit_predict(X_moons)
runtimes_moons['K-Means'] = time.time() - start

# OPTICS
start = time.time()
optics = OPTICS(min_samples=5, xi=0.05, min_cluster_size=0.05)
results_moons['OPTICS'] = optics.fit_predict(X_moons)
runtimes_moons['OPTICS'] = time.time() - start

# Hierarchical
start = time.time()
hierarchical = AgglomerativeClustering(n_clusters=2, linkage='ward')
results_moons['Hierarchical'] = hierarchical.fit_predict(X_moons)
runtimes_moons['Hierarchical'] = time.time() - start

print("✓ All algorithms completed")

In [ ]:
# Visualize side-by-side comparison
viz.plot_algorithm_comparison(
    X_moons,
    results_moons,
    title="Algorithm Comparison: Moons Dataset (Non-Convex Shapes)"
)
plt.show()

### Quantitative Metrics for Moons Dataset

In [ ]:
# Calculate silhouette scores
print("\n=== Moons Dataset Metrics ===")
print("\nSilhouette Scores (higher is better, range: [-1, 1]):")
for alg_name, labels in results_moons.items():
    # Only calculate if we have at least 2 clusters
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2:
        # For algorithms with noise, exclude noise points
        if -1 in labels:
            mask = labels != -1
            score = silhouette_score(X_moons[mask], labels[mask])
        else:
            score = silhouette_score(X_moons, labels)
        print(f"  {alg_name:15s}: {score:.3f}")
    else:
        print(f"  {alg_name:15s}: N/A (insufficient clusters)")

print("\nRuntime (seconds):")
for alg_name, runtime in runtimes_moons.items():
    print(f"  {alg_name:15s}: {runtime:.4f}s")

print("\nCluster Statistics:")
for alg_name, labels in results_moons.items():
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = np.sum(labels == -1)
    print(f"  {alg_name:15s}: {n_clusters} clusters, {n_noise} noise points")

**Observations:**
- **DBSCAN** correctly identifies the two moon-shaped clusters
- **K-Means** fails because it assumes spherical clusters
- **OPTICS** performs similarly to DBSCAN (it's an extension of DBSCAN)
- **Hierarchical** struggles with non-convex shapes depending on linkage method

## Part 3: Comparison on Nested Circles

The circles dataset contains concentric circles. This tests how algorithms handle nested cluster structures.

In [ ]:
# Generate circles dataset
X_circles, _ = make_circles(n_samples=300, noise=0.05, factor=0.5, random_state=42)

print(f"Dataset shape: {X_circles.shape}")

In [ ]:
# Run all algorithms
results_circles = {}
runtimes_circles = {}

# DBSCAN
start = time.time()
dbscan = DBSCAN(eps=0.2, min_pts=5)
results_circles['DBSCAN'] = dbscan.fit_predict(X_circles)
runtimes_circles['DBSCAN'] = time.time() - start

# K-Means
start = time.time()
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
results_circles['K-Means'] = kmeans.fit_predict(X_circles)
runtimes_circles['K-Means'] = time.time() - start

# OPTICS
start = time.time()
optics = OPTICS(min_samples=5, xi=0.05, min_cluster_size=0.05)
results_circles['OPTICS'] = optics.fit_predict(X_circles)
runtimes_circles['OPTICS'] = time.time() - start

# Hierarchical
start = time.time()
hierarchical = AgglomerativeClustering(n_clusters=2, linkage='ward')
results_circles['Hierarchical'] = hierarchical.fit_predict(X_circles)
runtimes_circles['Hierarchical'] = time.time() - start

print("✓ All algorithms completed")

In [ ]:
# Visualize comparison
viz.plot_algorithm_comparison(
    X_circles,
    results_circles,
    title="Algorithm Comparison: Circles Dataset (Nested Structures)"
)
plt.show()

In [ ]:
# Calculate metrics
print("\n=== Circles Dataset Metrics ===")
print("\nSilhouette Scores:")
for alg_name, labels in results_circles.items():
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters >= 2:
        if -1 in labels:
            mask = labels != -1
            score = silhouette_score(X_circles[mask], labels[mask])
        else:
            score = silhouette_score(X_circles, labels)
        print(f"  {alg_name:15s}: {score:.3f}")
    else:
        print(f"  {alg_name:15s}: N/A (insufficient clusters)")

print("\nRuntime (seconds):")
for alg_name, runtime in runtimes_circles.items():
    print(f"  {alg_name:15s}: {runtime:.4f}s")

## Part 4: Comparison on Varying Density Clusters

This dataset contains clusters with different densities. This is a challenging case for many algorithms.

In [ ]:
# Generate varying density dataset
np.random.seed(42)
X_dense = np.random.randn(100, 2) * 0.3 + [0, 0]
X_sparse = np.random.randn(100, 2) * 0.8 + [3, 3]
X_varying = np.vstack([X_dense, X_sparse])

print(f"Dataset shape: {X_varying.shape}")

In [ ]:
# Run all algorithms
results_varying = {}

# DBSCAN - struggles with varying density
dbscan = DBSCAN(eps=0.5, min_pts=5)
results_varying['DBSCAN'] = dbscan.fit_predict(X_varying)

# K-Means
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
results_varying['K-Means'] = kmeans.fit_predict(X_varying)

# OPTICS - better at varying density
optics = OPTICS(min_samples=5, xi=0.05, min_cluster_size=0.05)
results_varying['OPTICS'] = optics.fit_predict(X_varying)

# Hierarchical
hierarchical = AgglomerativeClustering(n_clusters=2, linkage='ward')
results_varying['Hierarchical'] = hierarchical.fit_predict(X_varying)

print("✓ All algorithms completed")

In [ ]:
# Visualize comparison
viz.plot_algorithm_comparison(
    X_varying,
    results_varying,
    title="Algorithm Comparison: Varying Density Clusters"
)
plt.show()

**Observations:**
- **DBSCAN** struggles because a single eps value can't capture both densities
- **OPTICS** handles varying densities better (this is its main advantage over DBSCAN)
- **K-Means** and **Hierarchical** can separate the clusters but don't capture density differences

## Part 5: Failure Cases for Each Algorithm

Let's examine specific scenarios where each algorithm fails.

### Failure Case 1: K-Means on Non-Convex Shapes

K-Means assumes spherical clusters and uses Euclidean distance to centroids. It fails on non-convex shapes.

In [ ]:
# Create elongated clusters
X_elongated = np.vstack([
    np.random.randn(100, 2) * [2, 0.3] + [0, 0],
    np.random.randn(100, 2) * [2, 0.3] + [0, 3]
])

kmeans_fail = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_kmeans_fail = kmeans_fail.fit_predict(X_elongated)

dbscan_success = DBSCAN(eps=0.5, min_pts=5)
labels_dbscan_success = dbscan_success.fit_predict(X_elongated)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.scatter(X_elongated[:, 0], X_elongated[:, 1], c=labels_kmeans_fail, cmap='viridis')
ax1.set_title('K-Means FAILS on Elongated Clusters')
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')

ax2.scatter(X_elongated[:, 0], X_elongated[:, 1], c=labels_dbscan_success, cmap='Spectral')
ax2.set_title('DBSCAN Succeeds')
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("K-Means incorrectly splits the elongated clusters vertically.")

### Failure Case 2: DBSCAN on Varying Density

DBSCAN uses a single eps parameter, so it struggles when clusters have different densities.

In [ ]:
# Already demonstrated above with varying density dataset
print("DBSCAN Failure: Varying Density")
print("- With small eps: Dense cluster is split, sparse cluster becomes noise")
print("- With large eps: Sparse cluster is captured, but dense cluster merges")
print("- Solution: Use OPTICS which handles varying densities")

### Failure Case 3: Hierarchical on Large Datasets

Hierarchical clustering has O(n²) or O(n³) complexity, making it slow on large datasets.

In [ ]:
# Compare runtime on increasing dataset sizes
sizes = [100, 500, 1000, 2000]
times_dbscan = []
times_hierarchical = []

for n in sizes:
    X_test = np.random.randn(n, 2)
    
    # DBSCAN
    start = time.time()
    DBSCAN(eps=0.5, min_pts=5).fit_predict(X_test)
    times_dbscan.append(time.time() - start)
    
    # Hierarchical
    start = time.time()
    AgglomerativeClustering(n_clusters=3).fit_predict(X_test)
    times_hierarchical.append(time.time() - start)

plt.figure(figsize=(10, 6))
plt.plot(sizes, times_dbscan, 'o-', label='DBSCAN', linewidth=2)
plt.plot(sizes, times_hierarchical, 's-', label='Hierarchical', linewidth=2)
plt.xlabel('Dataset Size (n)')
plt.ylabel('Runtime (seconds)')
plt.title('Runtime Comparison: DBSCAN vs Hierarchical')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Hierarchical clustering becomes impractical for large datasets (n > 10,000).")

### Failure Case 4: All Algorithms on High-Dimensional Data

The "curse of dimensionality" affects all distance-based clustering algorithms.

In [ ]:
# Generate high-dimensional data
from sklearn.datasets import make_classification

X_high_dim, _ = make_classification(
    n_samples=200,
    n_features=50,  # 50 dimensions
    n_informative=10,
    n_redundant=10,
    n_clusters_per_class=1,
    n_classes=3,
    random_state=42
)

print(f"High-dimensional dataset shape: {X_high_dim.shape}")
print("\nIn high dimensions:")
print("- Distance metrics become less meaningful")
print("- All points appear equidistant")
print("- DBSCAN's eps parameter is hard to tune")
print("- K-Means still works but may need many clusters")
print("\nSolution: Use dimensionality reduction (PCA, t-SNE) before clustering")

## Part 6: Summary and Decision Guide

### When to Use Each Algorithm

| Algorithm | Best For | Avoid When |
|-----------|----------|------------|
| **DBSCAN** | - Arbitrary cluster shapes<br>- Noise detection<br>- Spatial data<br>- Unknown number of clusters | - Varying density clusters<br>- High-dimensional data<br>- Need deterministic cluster assignment |
| **K-Means** | - Spherical clusters<br>- Known number of clusters<br>- Large datasets (fast)<br>- Need cluster centers | - Non-convex shapes<br>- Varying cluster sizes<br>- Outliers present<br>- Unknown k |
| **OPTICS** | - Varying density clusters<br>- Hierarchical structure<br>- Exploratory analysis | - Very large datasets (slow)<br>- Need simple parameters<br>- Real-time applications |
| **Hierarchical** | - Small datasets<br>- Need dendrogram<br>- Nested clusters | - Large datasets (O(n²) or O(n³))<br>- Need fast results<br>- Arbitrary shapes |

### Key Takeaways

1. **No algorithm is universally best** - choose based on your data characteristics
2. **DBSCAN excels at arbitrary shapes** but struggles with varying densities
3. **K-Means is fast and simple** but assumes spherical clusters
4. **OPTICS extends DBSCAN** to handle varying densities
5. **Hierarchical provides structure** but doesn't scale well
6. **Always visualize results** - metrics alone can be misleading
7. **Consider preprocessing** - dimensionality reduction, normalization, outlier removal

### Recommended Reading

- Paper Section 2: Related Work [p.227]
- `docs/06_algorithm_comparison.md` for detailed comparison
- `docs/04_parameter_tuning.md` for DBSCAN parameter selection

## Exercises

### Exercise 1: Algorithm Selection

For each scenario, which algorithm would you choose and why?

1. Clustering customer locations for retail store placement (geographic data)
2. Grouping similar documents in a text corpus (high-dimensional TF-IDF vectors)
3. Identifying anomalies in sensor readings
4. Segmenting an image into regions

### Exercise 2: Parameter Tuning

Try different parameter values for DBSCAN on the moons dataset:
- eps = [0.1, 0.2, 0.3, 0.5]
- min_pts = [3, 5, 10]

Which combination works best? Why?

### Exercise 3: Create Your Own Failure Case

Design a dataset where DBSCAN performs well but K-Means fails completely.
Visualize both results and explain why.

## Next Steps

- **Spatial Clustering**: See `08_spatial_clustering.ipynb` for real-world geographic data
- **Parameter Tuning**: See `06_parameter_tuning.ipynb` for DBSCAN parameter selection
- **Theory**: See `docs/06_algorithm_comparison.md` for detailed algorithm comparison